# Notebook 02 — Graph Construction

This is the most technically novel part of the project, so I want to walk through it carefully. The core idea is: instead of feeding raw text into a model, we convert each Arabic utterance into a **phoneme-level graph** where:
- **Nodes** are phonemes (IPA symbols)
- **Edges** connect adjacent phonemes (i→i+1) and skip-one pairs (i→i+2) to capture coarticulation
- **Node features** are 8-dim one-hot phoneme class + 1-dim positional encoding

The hypothesis: dialectal phonological variation (like Gulf ث→s, or Egyptian ق→ʔ) is more systematically captured at the phoneme level than at the subword token level.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import torch

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

DIALECT_NAMES = ['Gulf', 'Egyptian', 'Levantine', 'Maghrebi', 'Iraqi']
COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
print('Imports OK')

## 1. Phoneme extraction

The `CAMeLProcessor` class handles phoneme extraction. If `camel-tools` is installed, it uses Arabic morphological analysis + phoneme lookup. If not (the common case in a fresh environment), it falls back to a character-level Arabic→IPA mapping and logs a warning.

In [ ]:
from data.camel_pipeline import CAMeLProcessor, ARABIC_TO_PHONEME

processor = CAMeLProcessor()
print(f'Using CAMeL Tools: {processor._use_camel}')
if not processor._use_camel:
    print('Running in character-level fallback mode.')

# Side-by-side comparison of dialectally distinctive phrases
examples = [
    ('Gulf',      'شلونك اليوم'),    # sh-l-w-n-k: guttural-heavy
    ('Egyptian',  'ازيك النهارده'),  # q→? replacement
    ('Levantine', 'كيفك هلق'),       # ق retention (sometimes)
    ('Maghrebi',  'كيداير واش'),     # Berber substrate words
    ('Iraqi',     'شلونك شكو ماكو'),  # distinctive vocabulary
]

print('\n' + '='*60)
print(f'  {"Dialect":<12} {"Text":<22} Phonemes')
print('='*60)
for dialect, text in examples:
    phonemes = processor.text_to_phonemes(text)
    print(f'  {dialect:<12} {text:<22} {phonemes}')

## 2. Text → Phoneme → Graph

Now let's trace the full pipeline from a raw Arabic string to a PyTorch Geometric `Data` object. I'll use one sentence from each dialect and show the transformation step by step.

In [ ]:
from data.camel_pipeline import text_to_phoneme_graph
from data.graph_construction import phonemes_to_pyg_graph, add_positional_features, normalize_graph

gulf_text = 'شلونك اليوم'

# Step 1: Extract phonemes and edges
phonemes, edges, features = text_to_phoneme_graph(gulf_text, processor)
print(f'Text:     {gulf_text}')
print(f'Phonemes: {phonemes}  ({len(phonemes)} nodes)')
print(f'Edges:    {edges}  ({len(edges)} edges)')
print(f'Features: {len(features)} feature vectors, dim={len(features[0]) if features else 0}')

# Step 2: Build PyG Data object
graph = phonemes_to_pyg_graph(phonemes, edges, features, label=0)
graph = add_positional_features(graph)
graph = normalize_graph(graph)

print(f'\nPyG Data object:')
print(f'  x (node features): {graph.x.shape}  — dtype {graph.x.dtype}')
print(f'  edge_index:        {graph.edge_index.shape}')
print(f'  y (label):         {graph.y.item()} ({DIALECT_NAMES[graph.y.item()]})')
print(f'\nFirst 3 node feature vectors:')
for i, (ph, feat) in enumerate(zip(phonemes[:3], graph.x[:3])):
    print(f'  [{i}] phoneme={ph!r:6s}  features={feat.numpy().round(3)}')

## 3. Graph visualization

Let's visualize the phoneme graph for all 5 dialect examples. The key structural differences between dialects show up as:
- Different node counts (phoneme sequence length varies by dialect)
- Different connectivity density (affects how much context each node aggregates)

In [ ]:
try:
    import networkx as nx
    _NX = True
except ImportError:
    _NX = False
    print('networkx not installed — skipping visualization')

if _NX:
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    examples = [
        ('Gulf',      'شلونك اليوم',   0),
        ('Egyptian',  'ازيك النهارده', 1),
        ('Levantine', 'كيفك هلق',      2),
        ('Maghrebi',  'كيداير واش',    3),
        ('Iraqi',     'شلونك شكو ماكو', 4),
    ]

    for ax, (dialect, text, label) in zip(axes, examples):
        phonemes, edges, features = text_to_phoneme_graph(text, processor)
        G = nx.DiGraph()
        for i, ph in enumerate(phonemes):
            G.add_node(i, label=ph)
        for src, dst in edges:
            G.add_edge(src, dst)

        # Linear layout: phonemes in sequence order
        pos = {i: (i, 0) for i in range(len(phonemes))}
        node_labels = {i: phonemes[i] for i in range(len(phonemes))}

        nx.draw_networkx(G, pos=pos, labels=node_labels, ax=ax,
                         node_color=COLORS[label], node_size=400,
                         font_size=7, arrows=True, arrowsize=10,
                         with_labels=True, font_color='white')
        ax.set_title(f'{dialect}\n"{text}"', fontsize=9)
        ax.axis('off')

    plt.suptitle('Phoneme Graphs (one per dialect)', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig('../results/phoneme_graphs.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Node feature structure

Each phoneme node has a 9-dimensional feature vector:
- Dims 0–7: one-hot encoding of phoneme class (vowel/consonant/stop/fricative/etc.)
- Dim 8: normalized position in sequence (0 = first phoneme, 1 = last)

Here's a visual breakdown of what these features look like for a few phonemes.

In [ ]:
phonemes, edges, features = text_to_phoneme_graph('شلونك', processor)
graph = phonemes_to_pyg_graph(phonemes, edges, features, label=0)
graph = add_positional_features(graph)

x = graph.x.numpy()
n_nodes = x.shape[0]

fig, ax = plt.subplots(figsize=(10, 3))
im = ax.imshow(x, aspect='auto', cmap='Blues', vmin=0, vmax=1)

ax.set_yticks(range(n_nodes))
ax.set_yticklabels([f'[{i}] {phonemes[i]}' for i in range(n_nodes)], fontsize=10)
ax.set_xticks(range(9))
ax.set_xticklabels(
    ['class0', 'class1', 'class2', 'class3', 'class4', 'class5', 'class6', 'class7', 'pos'],
    rotation=30, ha='right', fontsize=8
)
ax.set_title('Node feature matrix — شلونك')
plt.colorbar(im, ax=ax, label='Feature value')
plt.tight_layout()
plt.show()

print('Feature matrix (raw):')
print(x.round(3))

## 5. Graph statistics across the dataset

How do the graphs differ in size and density across dialects? This matters because the GAT's aggregation behavior changes based on graph structure.

In [ ]:
from data.dataset import ArabicDialectGraphDataset
from models.graph_utils import compute_graph_stats

# Use the pre-built dataset (synthetic if graphs/ doesn't exist yet)
train_ds = ArabicDialectGraphDataset(root='../data/graphs', split='train')
stats = compute_graph_stats(train_ds)

# Plot node count distribution per dialect
node_counts_by_dialect = {d: [] for d in DIALECT_NAMES}
for data in train_ds:
    label = int(data.y.item())
    node_counts_by_dialect[DIALECT_NAMES[label]].append(int(data.num_nodes))

fig, ax = plt.subplots(figsize=(9, 4))
for i, dialect in enumerate(DIALECT_NAMES):
    ax.hist(node_counts_by_dialect[dialect], bins=20, alpha=0.55,
            color=COLORS[i], label=dialect)
ax.set_xlabel('Number of phoneme nodes')
ax.set_ylabel('Count')
ax.set_title('Graph size distribution (train split)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'\nMean graph size per dialect:')
for dialect, counts in node_counts_by_dialect.items():
    print(f'  {dialect:<12}: {np.mean(counts):.1f} nodes')

## 6. Edge construction: adjacency vs. coarticulation edges

For a sequence of N phonemes, we add two types of edges:
1. **Sequential** (i→i+1): basic phoneme adjacency
2. **Coarticulation** (i→i+2): phonemes two steps apart influence each other acoustically

Both are bidirectional. The total edges for a sequence of N phonemes = 2*(N-1) + 2*(N-2) = 4N - 6.

In [ ]:
def compute_edge_counts(n_nodes):
    sequential   = 2 * max(n_nodes - 1, 0)
    coarticulation = 2 * max(n_nodes - 2, 0)
    return sequential, coarticulation

ns = range(3, 20)
seq_counts  = [compute_edge_counts(n)[0] for n in ns]
coart_counts= [compute_edge_counts(n)[1] for n in ns]
total_counts= [s + c for s, c in zip(seq_counts, coart_counts)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.fill_between(ns, seq_counts, alpha=0.4, color='#4C72B0', label='Sequential (i→i+1)')
ax.fill_between(ns, total_counts, seq_counts, alpha=0.4, color='#DD8452', label='Coarticulation (i→i+2)')
ax.plot(ns, total_counts, 'k-', lw=1.5, label='Total edges')
ax.set_xlabel('Number of phoneme nodes')
ax.set_ylabel('Edge count')
ax.set_title('Edge count by graph size')
ax.legend()
plt.tight_layout()
plt.show()

# Show for a real example
phonemes, edges, _  = text_to_phoneme_graph('شلونك اليوم', processor)
seq_e  = sum(1 for s,d in edges if abs(s-d) == 1)
coart_e= sum(1 for s,d in edges if abs(s-d) == 2)
print(f'Example: \'شلونك اليوم\' → {len(phonemes)} phonemes, {seq_e} sequential edges, {coart_e} coarticulation edges')